In [ ]:
#@title ⚙️ نصب و دانلود
import base64
encoded_text = "Q3JlYXRlIGJ5IDogYWlnb2xkZW4="
decoded_text = base64.b64decode(encoded_text.encode()).decode()
print(decoded_text)
print('='*30)

!pip install -q -U google-genai yt-dlp pysrt pydub
!pip install -q omnivoice
!pip install -q torchaudio --extra-index-url https://download.pytorch.org/whl/cu128
!sudo apt-get update -qq
!sudo apt-get install -y -qq ffmpeg rubberband-cli

import torch
from omnivoice import OmniVoice

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
dtype  = torch.float16 if torch.cuda.is_available() else torch.float32
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)')

print('در حال دانلود و لود مدل ...')
omni_model = OmniVoice.from_pretrained('k2-fsa/OmniVoice', device_map=device, dtype=dtype)
print('✅ همه چیز آماده‌ست، مدل لود شد!')


In [ ]:
#@title 🔑 تنظیمات کلی پروژه
import os

#@markdown ##### 🔑 یک کلید دسترسی دریافت کنید
#@markdown ##### https://aistudio.google.com/api-keys
GOOGLE_API_KEY = '' #@param {type:"string"}
#@markdown ---
#@markdown #### 🌎 زبان مقصد برای ترجمه
Target_Language = 'fa' #@param ["fa", "en", "ar", "de", "fr", "es", "it", "tr", "ru", "zh-Hans", "ja", "ko", "hi", "pl", "pt-BR"]
#@markdown ---
#@markdown #### ✨ لحن گویندگی (برای ترجمه و لحن متن)
Tone = 'عامیانه / گفتاری' #@param ["عامیانه / گفتاری", "رسمی / اخبار / کتابی", "آموزشی / آکادمیک", "پرانرژی / تبلیغاتی", "طبیعی / پیش‌فرض"]
#@markdown ---
#@markdown #### 🎚️ صدای اصلی ویدیو
Keep_Original_Audio = False #@param {type:"boolean"}
Original_Audio_Volume = 0.1 #@param {type:"slider", min:0, max:1, step:0.05}
#@markdown ---
#@markdown #### ⏱️ زمان‌بندی و سینک
Allow_Timeline_Stretch = False #@param {type:"boolean"}
Max_Stretch_Percent = 10 #@param {type:"slider", min:0, max:25, step:1}
Balance_Speed_Across_Segments = True #@param {type:"boolean"}
Max_Speed_Factor = 1.4 #@param {type:"slider", min:1.0, max:2.0, step:0.1}

# مقادیر ثابت OmniVoice (بدون کنترل روی صفحه)
OmniVoice_Steps = 32
OmniVoice_Speed = 1.0

os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY
print('✅ تنظیمات ثبت شد')
print(f'🌐 زبان مقصد: {Target_Language}')
print(f'🎭 لحن بیان: {Tone}')
print(f'🎚️ صدای اصلی: {"حفظ شود" if Keep_Original_Audio else "حذف شود"} (حجم {Original_Audio_Volume})')
print(f'⏱️ کشش زمان‌بندی: {"فعال" if Allow_Timeline_Stretch else "غیرفعال"} ({Max_Stretch_Percent}%)')
print(f'⚡ تعادل سرعت: {"فعال" if Balance_Speed_Across_Segments else "غیرفعال"} (حداکثر {Max_Speed_Factor}x)')


In [ ]:
#@title ####📂 **Select video**
import subprocess, os, glob, shutil
from google.colab import files

for f in (glob.glob('input_video*') + glob.glob('yt_audio*') +
          ['audio.mp3', 'audio.srt', 'audio_translated.srt',
           'dubbed_audio.mp3', 'final_dubbed_video.mp4', 'raw_segments_combined.mp3']):
    if os.path.exists(f):
        os.remove(f)
if os.path.exists('dubbing_project'):
    shutil.rmtree('dubbing_project')
if os.path.exists('audio_chunks'):
    shutil.rmtree('audio_chunks')

Upload_Method = 'فایل از دستگاه' #@param ["یوتیوب", "فایل از دستگاه"]
YT_Link = '' #@param {type:"string"}

AUDIO_FILE = None
VIDEO_FILE = None
SRT_FILE = None
SUBTITLE_READY = False

# پارامترهای ضد بوت برای yt-dlp
YT_EXTRA_ARGS = ['--extractor-args', 'youtube:player_client=android,mweb,ios']

if Upload_Method == 'فایل از دستگاه':
    print('📁 فایل ویدیو یا صدا را آپلود کنید:')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('❌ فایلی آپلود نشد!')
    fname = list(uploaded.keys())[0]
    ext = fname.rsplit('.', 1)[-1].lower()
    if ext in ['mp4', 'mkv', 'avi', 'mov', 'webm']:
        VIDEO_FILE = 'input_video.mp4'
        os.rename(fname, VIDEO_FILE)
        res = subprocess.run(['ffmpeg', '-y', '-i', VIDEO_FILE, '-vn', '-acodec', 'libmp3lame',
                              '-ab', '64k', '-ac', '1', 'audio.mp3'], capture_output=True, text=True)
        if res.returncode != 0:
            raise RuntimeError(f'❌ استخراج صدا از فایل آپلودی با خطا مواجه شد:\n{res.stderr}')
        AUDIO_FILE = 'audio.mp3'
        print('✅ ویدیو آپلود شد، صدا استخراج شد')
    else:
        mp3_name = 'audio.mp3'
        res = subprocess.run(['ffmpeg', '-y', '-i', fname, '-acodec', 'libmp3lame',
                              '-ab', '64k', '-ac', '1', mp3_name], capture_output=True, text=True)
        if res.returncode != 0:
            raise RuntimeError(f'❌ تبدیل فایل صوتی با خطا مواجه شد:\n{res.stderr}')
        AUDIO_FILE = mp3_name
        print('✅ فایل صوتی آماده شد')

else:
    if not YT_Link.strip():
        raise ValueError('❌ لینک یوتیوب وارد نشده است!')

    print('🔍 بررسی زیرنویس ویدیو...')
    check = subprocess.run(['yt-dlp', '--list-subs'] + YT_EXTRA_ARGS + [YT_Link], capture_output=True, text=True)
    if check.returncode != 0:
        # تلاش مجدد با کلاینت جایگزین
        check = subprocess.run(['yt-dlp', '--list-subs', '--extractor-args', 'youtube:player_client=tv,mweb', YT_Link], capture_output=True, text=True)
        if check.returncode != 0:
            raise RuntimeError(f'❌ دسترسی به لینک یوتیوب امکان‌پذیر نیست یا ویدیو دارای محدودیت است:\n{check.stderr}')
        else:
            YT_EXTRA_ARGS = ['--extractor-args', 'youtube:player_client=tv,mweb']

    has_sub = 'has no subtitles' not in check.stdout and 'Language' in check.stdout

    if has_sub:
        print('✅ زیرنویس یافت شد، در حال دانلود...')
        res_sub = subprocess.run(['yt-dlp', '--write-subs', '--write-auto-subs',
                                  '--sub-format', 'srt', '--convert-subs', 'srt',
                                  '--skip-download', '--output', 'yt_audio.%(ext)s'] + YT_EXTRA_ARGS + [YT_Link],
                                 capture_output=True, text=True)
        srt_files = [f for f in os.listdir('.') if f.startswith('yt_audio') and f.endswith('.srt')]
        if srt_files:
            os.rename(srt_files[0], 'audio.srt')
            SUBTITLE_READY = True
            SRT_FILE = 'audio.srt'
            print('✅ زیرنویس دانلود شد!')

    print('🎬 در حال دانلود ویدیو...')
    res_vid = subprocess.run(['yt-dlp', '-f', 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
                              '-o', 'input_video.%(ext)s'] + YT_EXTRA_ARGS + [YT_Link], capture_output=True, text=True)
    if res_vid.returncode != 0:
        raise RuntimeError(f'❌ دانلود ویدیو از یوتیوب ناموفق بود:\n{res_vid.stderr}')

    vfiles = glob.glob('input_video.*')
    if not vfiles:
        raise FileNotFoundError('❌ فایل ویدیوی دانلود شده یافت نشد!')

    if not vfiles[0].endswith('.mp4'):
        res_cv = subprocess.run(['ffmpeg', '-y', '-i', vfiles[0], '-c', 'copy', 'input_video.mp4'], capture_output=True, text=True)
        if res_cv.returncode != 0:
            raise RuntimeError(f'❌ تبدیل ویدیو به mp4 با خطا مواجه شد:\n{res_cv.stderr}')
        os.remove(vfiles[0])

    VIDEO_FILE = 'input_video.mp4'
    print('✅  با موفقیت پیدا شد.')

    if not SUBTITLE_READY:
        print('🎵 در حال استخراج صدا...')
        res_aud = subprocess.run(['yt-dlp', '--extract-audio', '--audio-format', 'mp3',
                                  '--audio-quality', '5', '--postprocessor-args', 'ExtractAudio:-ac 1',
                                  '-o', 'audio.%(ext)s'] + YT_EXTRA_ARGS + [YT_Link], capture_output=True, text=True)
        if res_aud.returncode != 0 or not os.path.exists('audio.mp3'):
            raise RuntimeError(f'❌ استخراج صدای یوتیوب با خطا مواجه شد:\n{res_aud.stderr}')

        AUDIO_FILE = 'audio.mp3'
        size_mb = os.path.getsize(AUDIO_FILE) / 1024 / 1024
        print(f'✅ صدا استخراج شد ({size_mb:.1f} MB)')

print('\n▶️ فرایند با موفقیت انجام شد. حالا سلول بعدی را اجرا کنید.')


In [ ]:
#@title 🎙️ آپلود صدای خودتان برای کلون
from google.colab import files
from google import genai
from google.genai import types
import os

for f in ['ref_voice.wav', 'ref_voice.mp3', 'ref_voice.m4a', 'ref_voice.ogg']:
    if os.path.exists(f):
        os.remove(f)

#@markdown یک فایل صوتی کوتاه (حدود ۱۰ ثانیه، گفتار واضح و بدون نویز زمینه) از صدای خودتان آپلود کنید.
#@markdown متن گفته‌شده نیازی به وارد کردن دستی نداره؛ خودش با Gemini تشخیص داده می‌شه.
print('📁 لطفاً فایل صوتی صدای خودتان را آپلود کنید:')
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('❌ فایلی آپلود نشد!')

fname = list(uploaded.keys())[0]
ext = fname.rsplit('.', 1)[-1].lower()
REF_AUDIO_PATH = f'ref_voice.{ext}'
os.rename(fname, REF_AUDIO_PATH)

MIME_MAP = {'wav': 'audio/wav', 'mp3': 'audio/mp3', 'm4a': 'audio/mp4', 'ogg': 'audio/ogg'}
ref_mime_type = MIME_MAP.get(ext, 'audio/mp3')


def transcribe_ref_audio(audio_path, mime_type):
    client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])
    models = ['gemini-3.5-flash-lite', 'gemini-3.1-flash-lite']

    instruction = (
        "Transcribe EXACTLY the words spoken in this audio clip, in the original "
        "spoken language. Output ONLY the plain transcribed text — no timestamps, "
        "no labels, no quotation marks, no explanation, nothing else."
    )

    with open(audio_path, 'rb') as f:
        audio_bytes = f.read()

    for model_name in models:
        for attempt in range(1, 3):
            try:
                response = client.models.generate_content(
                    model=model_name,
                    contents=[
                        types.Part.from_bytes(data=audio_bytes, mime_type=mime_type),
                        instruction
                    ]
                )
                text = response.text.strip()
                if text:
                    return text, model_name
            except Exception as e:
                print(f'    ⚠️ خطا با {model_name} (تلاش {attempt}): {e}')

    return '', None


print('🧠 در حال تشخیص متن گفته‌شده در فایل صوتی...')
REF_TEXT, used_model = transcribe_ref_audio(REF_AUDIO_PATH, ref_mime_type)

print(f'✅ صدای مرجع ذخیره شد: {REF_AUDIO_PATH}')
if REF_TEXT:
    print(f'📝 متن تشخیص داده‌شده (با {used_model}): {REF_TEXT}')
else:
    print('⚠️ متن گفتار تشخیص داده نشد؛ خود OmniVoice از Whisper برای تشخیص خودکار استفاده خواهد کرد.')


In [ ]:
#@title ##🎙️ **شروع استخراج**
import os, sys, re, time, math, subprocess
import pysrt
from concurrent.futures import ThreadPoolExecutor, as_completed
from google import genai
from google.genai import types
from pydub import AudioSegment

TRANS_INSTRUCTION = (
    "CRITICAL MANDATE FOR AUDIO TRANSCRIBING:\n"
    "1. LANGUAGE STRICTNESS: Transcribe EXACTLY in the ORIGINAL spoken audio language. STRICTLY DO NOT TRANSLATE into English or any other language!\n"
    "2. TIMING & SYNC: Ensure timestamps match natural speech boundaries exactly in standard SRT format. Start and end times must accurately reflect when each full sentence or phrase is spoken.\n"
    "3. HUMAN SPEECH ONLY: Transcribe ONLY actual spoken human words. COMPLETELY IGNORE and DO NOT include any non-speech sounds or descriptions such as [applause], [کف زدن], [music], [موسیقی], [wind], [صدای باد], [laughter], [خنده], background noise, sound effects, or any bracketed/parenthetical sound descriptions. If a segment contains no spoken words (only ambient sound, applause, music, or noise), SKIP it entirely — do not create a subtitle entry for it.\n"
    "4. OUTPUT FORMAT: You MUST return the result in pure standard SRT format only. Example:\n"
    "1\n"
    "00:00:01,000 --> 00:00:04,500\n"
    "متن گفتار اینجا\n\n"
    "Do NOT use any other timestamp format like [0m0s...] or markdown."
)

def get_audio_duration(file_path):
    audio = AudioSegment.from_file(file_path)
    return len(audio) / 1000.0


def split_audio_on_silence(file_path, chunk_target_sec=300):
    duration = get_audio_duration(file_path)

    if duration > 150 * 60:
        print("\n" + "=" * 65)
        print("⚠️ هشدار مهم")
        print("مدت ویدیو بیشتر از ۱۵۰ دقیقه است.")
        print("برای اینکه بیشترین استفاده از مدل باکیفیت‌تر حفظ شود،")
        print("پیشنهاد می‌شود پیش از بارگذاری، ویدیو را به بخش‌های")
        print("حداکثر ۱۵۰ دقیقه‌ای تقسیم کنید.")
        print("این هشدار فقط جهت احتیاط است و پردازش متوقف نخواهد شد.")
        print("=" * 65 + "\n")

    if duration <= chunk_target_sec:
        return [(file_path, 0.0)]

    print(f"✂️ طول فایل {duration/60:.1f} دقیقه است. در حال برش هوشمند روی سکوت...")

    cmd = [
        'ffmpeg', '-i', file_path,
        '-af', 'silencedetect=noise=-30dB:d=0.5',
        '-f', 'null', '-'
    ]

    res = subprocess.run(cmd, capture_output=True, text=True)

    silences = re.findall(
        r'silence_start: ([\d\.]+)',
        res.stderr
    )

    silence_times = [float(s) for s in silences]

    os.makedirs('audio_chunks', exist_ok=True)

    chunks = []
    current_start = 0.0
    chunk_idx = 0

    while current_start < duration:

        target_time = current_start + chunk_target_sec

        if target_time >= duration:
            cut_time = duration
        else:
            candidates = [
                s for s in silence_times
                if current_start + (chunk_target_sec * 0.5) <= s <= current_start + chunk_target_sec + 60
            ]

            cut_time = (
                min(candidates, key=lambda x: abs(x - target_time))
                if candidates
                else target_time
            )

        chunk_path = f"audio_chunks/chunk_{chunk_idx:03d}.mp3"

        dur_segment = cut_time - current_start

        subprocess.run([
            'ffmpeg', '-y',
            '-ss', str(current_start),
            '-t', str(dur_segment),
            '-i', file_path,
            '-acodec', 'copy',
            chunk_path
        ], capture_output=True)

        chunks.append((chunk_path, current_start))

        print(
            f"   📌 تکه {chunk_idx+1}: "
            f"از {current_start:.1f}s تا {cut_time:.1f}s"
        )

        current_start = cut_time
        chunk_idx += 1

    return chunks


def convert_nonstandard_timestamp_to_srt(text):
    """تبدیل فرمت [ 0m0s610ms - 0m3s240ms ] به SRT استاندارد"""
    pattern = re.compile(
        r'\[\s*(\d+)m(\d+)s(\d+)ms\s*-\s*(\d+)m(\d+)s(\d+)ms\s*\]\s*(.*?)(?=\n\[|\Z)',
        re.DOTALL
    )

    blocks = []
    index = 1

    for match in pattern.finditer(text):
        m1, s1, ms1, m2, s2, ms2, content = match.groups()
        content = content.strip()
        if not content:
            continue

        start_total = int(m1) * 60 + int(s1) + int(ms1) / 1000
        end_total   = int(m2) * 60 + int(s2) + int(ms2) / 1000

        def to_srt_time(t):
            h = int(t // 3600)
            m = int((t % 3600) // 60)
            s = int(t % 60)
            ms = int(round((t - int(t)) * 1000))
            return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

        start_str = to_srt_time(start_total)
        end_str   = to_srt_time(end_total)

        blocks.append(f"{index}\n{start_str} --> {end_str}\n{content}")
        index += 1

    return "\n\n".join(blocks)


def is_valid_srt(text):
    if not text:
        return False
    if "1\n" in text and "-->" in text:
        return True
    if "00:" in text and "-->" in text:
        return True
    # تشخیص فرمت غیر استاندارد
    if re.search(r'\[\s*\d+m\d+s\d+ms\s*-\s*\d+m\d+s\d+ms\s*\]', text):
        return True
    return False


def try_generate_srt_text(client, model_name, audio_bytes, chunk_path, attempts=2):
    """با یک مدل مشخص، صدا رو مستقیماً به SRT تبدیل می‌کنه
    (روش استاندارد: مدل خودش متن رو به فرمت SRT برمی‌گردونه)"""

    for attempt in range(1, attempts + 1):

        try:
            print(
                f"🚀 در حال پردازش "
                f"{os.path.basename(chunk_path)} "
                f"با {model_name} "
                f"(تلاش {attempt})..."
            )

            response = client.models.generate_content(
                model=model_name,
                contents=[
                    types.Part.from_bytes(
                        data=audio_bytes,
                        mime_type='audio/mp3'
                    ),
                    TRANS_INSTRUCTION
                ]
            )

            srt_text = (response.text or '').strip()

            if is_valid_srt(srt_text):
                if re.search(r'\[\s*\d+m\d+s\d+ms\s*-\s*\d+m\d+s\d+ms\s*\]', srt_text):
                    print(f"    🔄 تبدیل فرمت غیر استاندارد به SRT...")
                    srt_text = convert_nonstandard_timestamp_to_srt(srt_text)
                if srt_text.strip():
                    return srt_text
            else:
                print(
                    f"    ⚠️ فرمت نامعتبر یا خالی از "
                    f"{model_name}: "
                    f"{srt_text[:120]}..."
                )

        except Exception as e:
            print(f"    ⚠️ خطا با {model_name}: {e}")
            if "503" in str(e) or "UNAVAILABLE" in str(e):
                time.sleep(5 * attempt)
            else:
                time.sleep(2 * attempt)

    return ""


def _offset_to_seconds(offset):
    """تبدیل start_offset/end_offset مدل gemini-3.5-transcribe به ثانیه،
    مستقل از این‌که خروجی timedelta، عدد یا رشته باشه"""
    if offset is None:
        return 0.0
    if hasattr(offset, 'total_seconds'):
        return offset.total_seconds()
    if hasattr(offset, 'seconds'):
        nanos = getattr(offset, 'nanos', 0) or 0
        return offset.seconds + nanos / 1e9
    if isinstance(offset, (int, float)):
        return float(offset)
    try:
        return float(str(offset).rstrip('s'))
    except Exception:
        return 0.0


def _seconds_to_srt_time(t):
    h = int(t // 3600)
    m = int((t % 3600) // 60)
    s = int(t % 60)
    ms = int(round((t - int(t)) * 1000))
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"


def try_transcribe_api(client, audio_bytes, chunk_path, max_words_per_line=12, max_line_sec=7.0, attempts=2):
    """تلاش برای رونویسی با مدل gemini-3.5-transcribe (تایم‌استمپ کلمه‌ای)
    و تبدیل کلمه‌به‌کلمه‌ی خروجی به فرمت SRT"""

    sentence_enders = ('.', '!', '?', '؟', '۔')

    for attempt in range(1, attempts + 1):

        try:
            print(
                f"🚀 در حال پردازش "
                f"{os.path.basename(chunk_path)} "
                f"با gemini-3.5-transcribe "
                f"(تلاش {attempt})..."
            )

            response = client.models.generate_content(
                model='gemini-3.5-transcribe',
                contents=[
                    types.Part.from_bytes(data=audio_bytes, mime_type='audio/mp3'),
                ],
                config=types.GenerateContentConfig(
                    audio_transcription_config=types.AudioTranscriptionConfig(
                        word_timestamp=True,
                    )
                ),
            )

            parts = getattr(response, 'parts', []) or []
            words = []
            if parts and getattr(parts[0], 'audio_transcription', None):
                words = getattr(parts[0].audio_transcription, 'words', []) or []

            if not words:
                print("    ⚠️ gemini-3.5-transcribe تایم‌استمپ کلمه‌ای برنگردوند.")
                continue

            blocks = []
            state = {'index': 1, 'cur_words': [], 'cur_start': None}

            def flush():
                if not state['cur_words']:
                    return
                start_sec = _offset_to_seconds(state['cur_start'])
                end_sec = _offset_to_seconds(state['cur_words'][-1].end_offset)
                text = ' '.join((w.word or '').strip() for w in state['cur_words']).strip()
                if text and end_sec > start_sec:
                    blocks.append(
                        f"{state['index']}\n"
                        f"{_seconds_to_srt_time(start_sec)} --> {_seconds_to_srt_time(end_sec)}\n"
                        f"{text}"
                    )
                    state['index'] += 1
                state['cur_words'] = []
                state['cur_start'] = None

            for w in words:
                if state['cur_start'] is None:
                    state['cur_start'] = w.start_offset
                state['cur_words'].append(w)

                word_text = (w.word or '').strip()
                ends_sentence = word_text.endswith(sentence_enders)
                too_long = len(state['cur_words']) >= max_words_per_line
                too_slow = (
                    _offset_to_seconds(w.end_offset) - _offset_to_seconds(state['cur_start'])
                ) >= max_line_sec

                if ends_sentence or too_long or too_slow:
                    flush()

            flush()

            srt_text = "\n\n".join(blocks)
            if srt_text.strip():
                return srt_text

        except Exception as e:
            print(f"    ⚠️ خطا با gemini-3.5-transcribe: {e}")
            if "503" in str(e) or "UNAVAILABLE" in str(e):
                time.sleep(5 * attempt)
            else:
                time.sleep(2 * attempt)

    return ""


def transcribe_chunk_with_fallback(chunk_info, use_rate_limit=False):
    chunk_path, offset_sec = chunk_info

    client = genai.Client(
        api_key=os.environ['GOOGLE_API_KEY']
    )

    with open(chunk_path, 'rb') as f:
        audio_bytes = f.read()

    # سطح ۱: gemini-3.8-flash
    srt_text = try_generate_srt_text(client, 'gemini-3.8-flash', audio_bytes, chunk_path)
    if srt_text:
        return srt_text, offset_sec

    # سطح ۲: gemini-3.5-transcribe (رونویسی کلمه‌به‌کلمه با تایم‌استمپ)
    srt_text = try_transcribe_api(client, audio_bytes, chunk_path)
    if srt_text:
        return srt_text, offset_sec

    # سطح ۳: مدل فعلی به‌عنوان fallback نهایی
    for model_name in ['gemini-3.5-flash-lite', 'gemini-3.1-flash-lite']:
        srt_text = try_generate_srt_text(client, model_name, audio_bytes, chunk_path, attempts=4)
        if srt_text:
            return srt_text, offset_sec

    return "", offset_sec


def shift_srt_timestamps(srt_text, offset_sec):

    lines = srt_text.strip().split('\n')

    new_lines = []

    pattern = re.compile(
        r'(\d{2}):(\d{2}):(\d{2})[,.](\d{3})\s*-->\s*'
        r'(\d{2}):(\d{2}):(\d{2})[,.](\d{3})'
    )

    def add_offset(h, m, s, ms):

        tot = (
            int(h) * 3600
            + int(m) * 60
            + int(s)
        ) + int(ms) / 1000.0 + offset_sec

        nh, rem = divmod(tot, 3600)
        nm, ns = divmod(rem, 60)

        nms = int(
            (ns - int(ns)) * 1000
        )

        return (
            f"{int(nh):02d}:"
            f"{int(nm):02d}:"
            f"{int(ns):02d},"
            f"{nms:03d}"
        )

    for line in lines:

        match = pattern.search(line)

        if match:

            start_str = add_offset(
                *match.groups()[:4]
            )

            end_str = add_offset(
                *match.groups()[4:]
            )

            new_lines.append(
                f"{start_str} --> {end_str}"
            )

        else:
            new_lines.append(line)

    return "\n".join(new_lines)


def analyze_wpm(srt_file):

    if not os.path.exists(srt_file):
        return

    subs = pysrt.open(
        srt_file,
        encoding='utf-8'
    )

    if not subs:
        return

    total_words = 0
    total_seconds = 0
    wpm_list = []
    high_speed_count = 0

    for sub in subs:

        words = len(
            sub.text.split()
        )

        duration = (
            sub.end.ordinal
            - sub.start.ordinal
        ) / 1000.0

        if duration <= 0:
            continue

        wpm = (
            words / duration
        ) * 60

        wpm_list.append(wpm)

        total_words += words
        total_seconds += duration

        if wpm > 150:
            high_speed_count += 1

    avg_wpm = (
        (total_words / total_seconds) * 60
        if total_seconds > 0
        else 0
    )

    print("\n" + "=" * 50)
    print("📊 **گزارش آنالیز سرعت گفتار (WPM Log)**")
    print(
        f"🔹 میانگین کل سرعت گفتار: "
        f"{avg_wpm:.1f} کلمه در دقیقه"
    )

    if avg_wpm <= 110:
        print("🟢 وضعیت: آهسته و شمرده (سبز)")
    elif avg_wpm <= 150:
        print("🔵 وضعیت: طبیعی و روان (آبی)")
    elif avg_wpm <= 180:
        print("🟡 وضعیت: سریع و متراکم (زرد)")
    else:
        print("🔴 وضعیت: بسیار سریع و فشرده (قرمز)")

    if high_speed_count > 0:

        percent = (
            high_speed_count / len(subs)
        ) * 100

        print(
            f"\n⚠️ **هشدار:** "
            f"{percent:.1f}% دیالوگ‌ها سرعت زرد یا قرمز دارند."
        )

        print(
            "👉 توجه: سلول ترجمه به‌صورت خودکار دیالوگ‌های پرریسک را "
            "تشخیص داده و خلاصه ترجمه می‌کند، نیازی به کار دستی نیست."
        )

    print("=" * 50 + "\n")


def fix_invalid_timings(subs_list):
    """اصلاح دو نوع خطای زمان‌بندی که ممکنه در خروجی Gemini پیش بیاد:
    ۱) دیالوگی که با دیالوگ قبلی تداخل زمانی داره یا حتی قبل از پایانش شروع می‌شه
       (مثلاً وقتی مدل در یکی از تکه‌های صوتی دچار هالوسینیشن/تکرار بشه و
       زمان‌بندیِ یه بخش قبلی رو دوباره برگردونه)
    ۲) دیالوگی که زمان پایانش قبل یا هم‌زمان با زمان شروعشه
    هر دو با در نظر گرفتن دیالوگ قبل و بعد اصلاح می‌شن، طوری که ترتیب شماره‌ها
    (و در نتیجه خود ویدیو) دست‌نخورده بمونه و فقط زمان‌بندی جمع‌وجور بشه."""

    MIN_GAP_MS = 50        # حداقل فاصله مجاز با دیالوگ قبل/بعد
    MIN_DURATION_MS = 300  # حداقل مدت زمان قابل‌قبول برای یک دیالوگ

    n = len(subs_list)
    fixed_count = 0

    for i, sub in enumerate(subs_list):
        start_ms = sub.start.ordinal
        end_ms = sub.end.ordinal
        changed = False

        # ۱) تداخل با دیالوگ قبلی: شروع این یکی قبل از پایان دیالوگ قبلی (یا خیلی نزدیکشه)
        if i > 0:
            prev_end_ms = subs_list[i - 1].end.ordinal
            if start_ms < prev_end_ms + MIN_GAP_MS:
                start_ms = prev_end_ms + MIN_GAP_MS
                changed = True

        # ۲) زمان پایان نامعتبر (قبل یا هم‌زمان با شروع) → بر اساس تعداد کلمات تخمین می‌زنیم
        if end_ms <= start_ms:
            word_count = max(1, len(sub.text.split()))
            estimated_ms = max(int((word_count / 150) * 60 * 1000), MIN_DURATION_MS)
            end_ms = start_ms + estimated_ms
            changed = True

        # سقف مجاز برای پایان: قبل از شروع دیالوگ بعدی (با فاصله امن) تا جلوشو نگیره
        if i < n - 1:
            next_start_ms = subs_list[i + 1].start.ordinal
            max_allowed_end_ms = next_start_ms - MIN_GAP_MS
            if max_allowed_end_ms > start_ms and end_ms > max_allowed_end_ms:
                end_ms = max_allowed_end_ms
                changed = True

        # تضمین نهایی مدت زمان مثبت
        if end_ms <= start_ms:
            end_ms = start_ms + MIN_DURATION_MS
            changed = True

        if changed:
            sub.start.ordinal = start_ms
            sub.end.ordinal = end_ms
            fixed_count += 1
            print(
                f"  🔧 زیرنویس شماره {sub.index}: زمان‌بندی نامعتبر/متداخل بود، "
                f"اصلاح شد به {sub.start} --> {sub.end}"
            )

    if fixed_count:
        print(f"\n🛠️ در مجموع {fixed_count} زیرنویس با زمان‌بندی نامعتبر/متداخل پیدا و اصلاح شد.\n")

    return fixed_count


if AUDIO_FILE is None or not os.path.exists(AUDIO_FILE):

    if VIDEO_FILE and os.path.exists(VIDEO_FILE):

        print(
            "🎵 در حال استخراج فایل صوتی از ویدیو "
            "جهت پردازش با Gemini..."
        )

        res = subprocess.run(
            [
                'ffmpeg', '-y',
                '-i', VIDEO_FILE,
                '-vn',
                '-acodec', 'libmp3lame',
                '-ab', '64k',
                '-ac', '1',
                'audio.mp3'
            ],
            capture_output=True,
            text=True
        )

        if res.returncode == 0:
            AUDIO_FILE = 'audio.mp3'


if AUDIO_FILE is None or not os.path.exists(AUDIO_FILE):

    print(
        '❌ ابتدا سلول ورودی را اجرا کنید '
        'یا مطمئن شوید فایل صوتی/ویدیویی موجود است!'
    )


else:

    if SUBTITLE_READY:

        print(
            '🔄 زیرنویس آماده یوتیوب نادیده گرفته می‌شود؛ '
            'با Gemini زیرنویس جدید استخراج می‌گردد...'
        )

    SRT_FILE = 'audio.srt'

    audio_duration = get_audio_duration(AUDIO_FILE)

    use_rate_limit = audio_duration >= 40 * 60

    chunks = split_audio_on_silence(
        AUDIO_FILE,
        chunk_target_sec=300
    )

    results = []

    with ThreadPoolExecutor(
        max_workers=min(4, len(chunks))
    ) as executor:

        futures = [
            executor.submit(
                transcribe_chunk_with_fallback,
                chunk,
                use_rate_limit
            )
            for chunk in chunks
        ]

        for future in as_completed(futures):
            results.append(
                future.result()
            )

    results.sort(
        key=lambda x: x[1]
    )

    all_subs = []

    global_index = 1

    for raw_srt, offset in results:

        if not raw_srt.strip():
            continue

        shifted = shift_srt_timestamps(
            raw_srt,
            offset
        )

        try:
            subs = pysrt.from_string(shifted)
        except Exception:
            print(f"⚠️ نتوانستم یکی از تکه‌ها را پارس کنم، رد می‌شود.")
            continue

        for sub in subs:

            sub.index = global_index
            global_index += 1

            all_subs.append(sub)

    if all_subs:
        print("\n🔎 در حال بررسی اعتبار زمان‌بندی زیرنویس‌ها...")
        fix_invalid_timings(all_subs)

    final_srt_blocks = [str(sub) for sub in all_subs]

    with open(
        SRT_FILE,
        'w',
        encoding='utf-8'
    ) as f:

        f.write(
            "\n\n".join(final_srt_blocks)
        )

    SUBTITLE_READY = True

    if final_srt_blocks:
        print(
            f'✅ زیرنویس جدید با Gemini تولید و '
            f'در {SRT_FILE} جایگزین شد.'
        )
        analyze_wpm(SRT_FILE)
    else:
        print('❌ هیچ زیرنویسی تولید نشد. لطفاً دوباره تلاش کنید.')


In [ ]:
#@title ##🌐 **ترجمه زیرنویس**
import pysrt, os, time, re, shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
from google import genai

# هر بار که این سلول اجرا بشه یعنی ترجمه‌ی جدیدی قراره انجام بشه
# (مثلاً تغییر زبان مقصد)، پس سگمنت‌های صوتیِ دوبله‌ی قبلی رو پاک می‌کنیم
# تا توی سلول «دوبله هوشمند» به‌اشتباه با زبان/ترجمه‌ی جدید ترکیب نشن
for _f in ['dubbed_audio.mp3', 'final_dubbed_video.mp4', 'raw_segments_combined.mp3']:
    if os.path.exists(_f):
        os.remove(_f)
if os.path.exists('dubbing_project'):
    shutil.rmtree('dubbing_project')

# اگر طول متن ترجمه‌شده از طول متن اصلی بیش از این نسبت بیشتر شود،
# آن دیالوگ به‌عنوان «پرریسک» علامت می‌خورد و دوباره به‌صورت خلاصه ترجمه می‌شود
Overflow_Ratio_Threshold = 1.3


def translate_batch(batch_subs, target_lang, enable_summary, tone_option):
    client = genai.Client(api_key=os.environ['GOOGLE_API_KEY'])
    models = ['gemini-3.5-flash-lite', 'gemini-3.1-flash-lite']

    texts = [f"{i+1}. {sub.text}" for i, sub in enumerate(batch_subs)]
    combined_text = "\n".join(texts)

    tone_prompts = {
        "عامیانه / گفتاری": "Use casual, everyday spoken phrasing with natural contractions, like talking to a friend.",
        "رسمی / اخبار / کتابی": "Use formal, precise written language suitable for news or literary narration.",
        "آموزشی / آکادمیک": "Use clear, structured, instructional phrasing suitable for an educational lecture.",
        "پرانرژی / تبلیغاتی": "Use energetic, punchy, exclamatory phrasing suitable for an advertisement.",
        "طبیعی / پیش‌فرض": "Use natural, balanced everyday phrasing."
    }
    tone_prompt = tone_prompts.get(tone_option, tone_prompts["طبیعی / پیش‌فرض"])

    summary_prompt = "Keep translations noticeably shorter and more concise while preserving the full meaning." if enable_summary else ""
    prompt = f"Translate the following subtitles into target language '{target_lang}'. {tone_prompt} {summary_prompt} Maintain line numbering strictly.\n\n{combined_text}"

    for model_name in models:
        try:
            res = client.models.generate_content(model=model_name, contents=prompt)
            lines = res.text.strip().split('\n')
            translated = [re.sub(r'^\d+\.\s*', '', line) for line in lines if line.strip()]
            if len(translated) == len(batch_subs):
                return translated
        except Exception:
            time.sleep(1)

    return [sub.text for sub in batch_subs]


def translate_all(subs_list, target_lang, enable_summary, tone_option, batch_size=20):
    """ترجمه موازی یک لیست از زیرنویس‌ها و بازگرداندن متن‌های ترجمه‌شده به همان ترتیب"""
    batches = [subs_list[i:i + batch_size] for i in range(0, len(subs_list), batch_size)]
    results = []
    with ThreadPoolExecutor(max_workers=5) as executor:
        futures = [
            executor.submit(translate_batch, b, target_lang, enable_summary, tone_option)
            for b in batches
        ]
        for future in futures:
            results.extend(future.result())
    return results


# مسیر فایل اصلی زیرنویس ثابت و مستقل از SRT_FILE، تا اجراهای مکرر همیشه از روی زیرنویس اصلی ترجمه کنن
ORIGINAL_SRT_FILE = 'audio.srt'

if os.path.exists(ORIGINAL_SRT_FILE):
    subs = pysrt.open(ORIGINAL_SRT_FILE, encoding='utf-8')
    original_texts = [sub.text for sub in subs]  # قبل از رونویسی، متن اصلی رو نگه می‌داریم

    print("🌐 مرحله ۱ از ۲: در حال ترجمه کامل زیرنویس...")
    translated_texts = translate_all(list(subs), Target_Language, False, Tone)

    # مرحله مقایسه: کدام دیالوگ‌ها نسبت به متن اصلی طولانی‌تر شدن و ریسک همپوشانی دارن
    risky_indices = [
        i for i, (orig, trans) in enumerate(zip(original_texts, translated_texts))
        if len(orig.strip()) > 0 and (len(trans.strip()) / len(orig.strip())) > Overflow_Ratio_Threshold
    ]

    if risky_indices:
        print(f"🔍 مقایسه طول دیالوگ‌ها انجام شد: {len(risky_indices)} از {len(subs)} دیالوگ ریسک همپوشانی دارند.")
        print("🌐 مرحله ۲ از ۲: در حال ترجمه خلاصه فقط برای دیالوگ‌های پرریسک...")

        risky_subs = [subs[i] for i in risky_indices]  # subs[i].text هنوز متن اصلیه (هنوز رونویسی نشده)
        risky_translated = translate_all(risky_subs, Target_Language, True, Tone)

        for idx, new_text in zip(risky_indices, risky_translated):
            translated_texts[idx] = new_text
    else:
        print("✅ مقایسه طول دیالوگ‌ها انجام شد: هیچ دیالوگی ریسک همپوشانی نداشت.")

    for i, t_text in enumerate(translated_texts):
        if i < len(subs):
            subs[i].text = t_text

    TRANSLATED_SRT = 'audio_translated.srt'
    subs.save(TRANSLATED_SRT, encoding='utf-8')
    SRT_FILE = TRANSLATED_SRT
    print(f"✅ ترجمه کامل شد و در {SRT_FILE} ذخیره شد. ({len(risky_indices)} دیالوگ به‌صورت خلاصه ترجمه شدند)")
else:
    print("⚠️ فایل اصلی زیرنویس (audio.srt) پیدا نشد.")


In [ ]:
#@title ##🎬 **دوبله هوشمند**
import os, subprocess, shutil
import pysrt
import torch, torchaudio
from pydub import AudioSegment
from google.colab import files as colab_files

#@markdown #### 🔊 دانلود نسخه خام هم‌زمان (بدون اعمال سرعت/سینک)
Export_Raw_Track = False #@param {type:"boolean"}

os.makedirs('dubbing_project/dubbed_segments', exist_ok=True)

STRETCH_FACTOR = 1.0


def generate_one_segment_omnivoice(i, text_to_speak, subtitle_duration, raw_path, natural_ms):
    """تولید صدای هر دیالوگ با صدای کلون‌شده‌ی خودتان (OmniVoice)"""
    if not text_to_speak or not text_to_speak.strip():
        AudioSegment.silent(duration=int(subtitle_duration * 1000)).export(raw_path, format='wav')
        natural_ms[i] = int(subtitle_duration * 1000)
        return

    for attempt in range(1, 4):
        try:
            kwargs = dict(
                text=text_to_speak,
                ref_audio=REF_AUDIO_PATH,
                num_step=int(OmniVoice_Steps),
                speed=float(OmniVoice_Speed),
            )
            if REF_TEXT.strip():
                kwargs['ref_text'] = REF_TEXT
            # اگر متن مرجع خالی باشد، خود OmniVoice با Whisper آن را تشخیص می‌دهد

            audio = omni_model.generate(**kwargs)
            audio_tensor = audio[0] if isinstance(audio[0], torch.Tensor) else torch.tensor(audio[0])
            if audio_tensor.dim() == 1:
                audio_tensor = audio_tensor.unsqueeze(0)
            torchaudio.save(raw_path, audio_tensor.cpu(), 24000)

            natural_ms[i] = len(AudioSegment.from_file(raw_path))
            print(f'  ✅ [{i+1}] خوانده شد: "{text_to_speak[:40]}..."')
            return
        except Exception as e:
            if attempt == 3:
                AudioSegment.silent(duration=int(subtitle_duration * 1000)).export(raw_path, format='wav')
                natural_ms[i] = int(subtitle_duration * 1000)
                print(f'  ⚠️ [{i+1}] خطا بعد از ۳ تلاش: {e}')


def run_dubbing():
    global STRETCH_FACTOR
    STRETCH_FACTOR = 1.0

    if 'REF_AUDIO_PATH' not in globals() or not os.path.exists(REF_AUDIO_PATH):
        print('❌ ابتدا سلول «آپلود صدای خودتان برای کلون» را اجرا کنید!')
        return

    srt_path = SRT_FILE if 'SRT_FILE' in globals() and SRT_FILE and os.path.exists(SRT_FILE) else 'audio_translated.srt'
    if not os.path.exists(srt_path):
        srt_path = 'audio.srt'

    if not os.path.exists(srt_path):
        print('❌ فایل زیرنویس یافت نشد!')
        return

    subs = pysrt.open(srt_path, encoding='utf-8')
    n = len(subs)

    windows_ms = [None] * n
    natural_ms = [None] * n
    raw_paths = [None] * n

    print(f'🚀 در حال گویندگی {n} دیالوگ با صدای شما (به‌صورت پیاپی روی GPU)...')

    for i, sub in enumerate(subs):
        start_ms = (sub.start.hours*3600 + sub.start.minutes*60 + sub.start.seconds)*1000 + sub.start.milliseconds
        end_ms = (sub.end.hours*3600 + sub.end.minutes*60 + sub.end.seconds)*1000 + sub.end.milliseconds
        subtitle_duration = (end_ms - start_ms) / 1000.0

        if i < n - 1:
            next_sub = subs[i+1]
            next_start_ms = (next_sub.start.hours*3600 + next_sub.start.minutes*60 + next_sub.start.seconds)*1000 + next_sub.start.milliseconds
            window_ms = next_start_ms - start_ms
        else:
            window_ms = end_ms - start_ms

        raw_path = f'dubbing_project/dubbed_segments/raw_{i+1}.wav'
        windows_ms[i] = window_ms
        raw_paths[i] = raw_path

        if os.path.exists(raw_path) and os.path.getsize(raw_path) > 100:
            try:
                natural_ms[i] = len(AudioSegment.from_file(raw_path))
                print(f'  ♻️ [{i+1}/{n}] قبلاً تولید شده بود، رد شد.')
                continue
            except Exception:
                pass

        generate_one_segment_omnivoice(i, sub.text, subtitle_duration, raw_path, natural_ms)

    if any(v is None for v in natural_ms):
        print("❌ برخی از دیالوگ‌ها با خطا مواجه شدند.")
        return

    if Export_Raw_Track:
        print('🔊 در حال ساخت تراک خام (بدون اعمال سرعت)...')
        combined = AudioSegment.empty()
        for rp in raw_paths:
            if os.path.exists(rp):
                combined += AudioSegment.from_file(rp)
        raw_track_file = 'raw_segments_combined.mp3'
        combined.export(raw_track_file, format='mp3', bitrate='192k')
        colab_files.download(raw_track_file)
        print(f'✅ تراک خام ذخیره و دانلود شد: {raw_track_file}')

    idx_borrowable = list(range(n - 1)) if n > 1 else []
    total_natural = sum(natural_ms[i] for i in idx_borrowable)
    total_window  = sum(windows_ms[i] for i in idx_borrowable)
    G = (total_natural / total_window) if total_window > 0 else 1.0

    if Allow_Timeline_Stretch and G > 1.0:
        max_stretch = 1 + (Max_Stretch_Percent / 100.0)
        STRETCH_FACTOR = min(G, max_stretch)
    if STRETCH_FACTOR > 1.0:
        windows_ms = [w * STRETCH_FACTOR if i in idx_borrowable else w for i, w in enumerate(windows_ms)]

    print(f"📐 نسبت طول دوبله به زمان اصلی (G): {G:.2f}  →  یعنی صدا حدود {(G-1)*100:.0f}٪ بیشتر از زمان اصلی فضا لازم داره")

    R = (total_natural / total_window) if total_window > 0 else 1.0
    base_factor = min(R, 1.3) if (Balance_Speed_Across_Segments and R > 1.0) else 1.0
    hard_cap = Max_Speed_Factor if Balance_Speed_Across_Segments else 1.4

    print("⚡ در حال تنظیم زمان‌بندی و سینک صدا...")
    for i in range(n):
        raw_path = raw_paths[i]
        final_path = f'dubbing_project/dubbed_segments/dub_{i+1}.wav'

        adjusted_ms = natural_ms[i] / base_factor
        local_factor = adjusted_ms / windows_ms[i] if windows_ms[i] > 0 else 1.0
        final_factor = max(1.0, min(base_factor * max(1.0, local_factor), hard_cap))

        if final_factor <= 1.001:
            shutil.copy(raw_path, final_path)
        else:
            try:
                if not os.path.exists(raw_path) or os.path.getsize(raw_path) < 100:
                    shutil.copy(raw_path, final_path)
                    continue

                res = subprocess.run([
                    'ffmpeg', '-y', '-i', raw_path,
                    '-filter:a', f'rubberband=tempo={final_factor:.3f}',
                    final_path
                ], capture_output=True, text=True)

                if res.returncode != 0:
                    res_alt = subprocess.run([
                        'ffmpeg', '-y', '-i', raw_path,
                        '-filter:a', f'atempo={final_factor:.3f}',
                        final_path
                    ], capture_output=True, text=True)
                    if res_alt.returncode != 0:
                        shutil.copy(raw_path, final_path)
            except Exception:
                shutil.copy(raw_path, final_path)

    print('✅ دوبله هوشمند با صدای شما تمام شد!')

run_dubbing()


In [ ]:
#@title ##🎞️ **ترکیب نهایی و دانلود**
import subprocess, os, glob, shutil
import pysrt
from pydub import AudioSegment
from google.colab import files
from IPython.display import display, HTML

try:
    STRETCH_FACTOR
except NameError:
    STRETCH_FACTOR = 1.0

# تأخیر جبرانی ثابت برای شروع هر دیالوگ دوبله‌شده
START_DELAY_MS = 150

if not VIDEO_FILE or not os.path.exists(VIDEO_FILE):
    subs = pysrt.open(SRT_FILE, encoding='utf-8')
    total_ms = (subs[-1].end.hours*3600 + subs[-1].end.minutes*60 +
                subs[-1].end.seconds)*1000 + subs[-1].end.milliseconds
    total_ms = int(total_ms * STRETCH_FACTOR)
    final_audio = AudioSegment.silent(duration=total_ms)
    for i, sub in enumerate(subs):
        seg_path = f'dubbing_project/dubbed_segments/dub_{i+1}.wav'
        if os.path.exists(seg_path):
            start_ms = int(((sub.start.hours*3600 + sub.start.minutes*60 + sub.start.seconds)*1000 + sub.start.milliseconds) * STRETCH_FACTOR) + START_DELAY_MS
            seg_audio = AudioSegment.from_file(seg_path)
            final_audio = final_audio.overlay(seg_audio, position=start_ms)
    output_file = 'dubbed_audio.mp3'
    final_audio.export(output_file, format='mp3', bitrate='192k')
    files.download(output_file)
else:
    subs = pysrt.open(SRT_FILE, encoding='utf-8')
    orig_tempo = 1/STRETCH_FACTOR if STRETCH_FACTOR > 1 else 1.0

    if Keep_Original_Audio:
        filter_parts = [f'[0:a]volume={Original_Audio_Volume},atempo={orig_tempo}[orig]']
    else:
        filter_parts = ['[0:a]volume=0[orig]']

    input_args = ['-i', VIDEO_FILE]
    valid_segs = []

    for i, sub in enumerate(subs):
        seg_path = f'dubbing_project/dubbed_segments/dub_{i+1}.wav'
        if os.path.exists(seg_path):
            start_ms = int(((sub.start.hours*3600 + sub.start.minutes*60 +
                        sub.start.seconds)*1000 + sub.start.milliseconds) * STRETCH_FACTOR) + START_DELAY_MS
            filter_parts.append(f'[{len(valid_segs)+1}:a]adelay={start_ms}|{start_ms}[a{i+1}]')
            input_args += ['-i', seg_path]
            valid_segs.append(i+1)

    if valid_segs:
        mix_inputs = '[orig]' + ''.join(f'[a{i}]' for i in valid_segs)
        filter_parts.append(f'{mix_inputs}amix=inputs={len(valid_segs)+1}:normalize=0[aout]')

        if STRETCH_FACTOR > 1.0:
            filter_parts.append(f'[0:v]setpts={STRETCH_FACTOR}*PTS[vout]')
            video_map = ['-map', '[vout]']
            vcodec = 'libx264'
        else:
            video_map = ['-map', '0:v']
            vcodec = 'copy'

        filter_complex = ';'.join(filter_parts)
        output_file = 'final_dubbed_video.mp4'
        cmd = ['ffmpeg', '-y'] + input_args + ['-filter_complex', filter_complex] + video_map + [
            '-map', '[aout]', '-c:v', vcodec, '-c:a', 'aac', output_file
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode == 0:
            files.download(output_file)

image_url = 'https://huggingface.co/Toolsai/dubtest/resolve/main/newgolden.png'
yt_url = 'https://youtube.com/@aigolden'
display(HTML(f'''
<div style="text-align:center;border:2px solid #e0e0e0;padding:15px;border-radius:12px;background:#f9f9f9;max-width:350px;margin:auto">
  <a href="{yt_url}" target="_blank">
    <img src="{image_url}" style="max-width:100%;border-radius:8px">
  </a>
  <p style="font-family:Vazir,sans-serif;margin-top:10px">برای آموزش‌های بیشتر ما را در یوتیوب دنبال کنید</p>
  <a href="{yt_url}" target="_blank" style="background:#FF0000;color:white;padding:10px 20px;border-radius:8px;text-decoration:none;font-weight:bold">🚀 دنبال کردن</a>
</div>
'''))
